In [1]:
from datasets import Dataset
# Чтение файла txt
with open("../models/train.txt", 'r') as file:
    lines = file.readlines()
    

# Преобразование данных в формат, подходящий для загрузки в datasets
dataset_dict = {'text': lines}
dataset = Dataset.from_dict(dataset_dict)

# Необязательно: указание типа данных текста
dataset = dataset.map(lambda example: {'text': example['text']}, batched=True)

Map:   0%|          | 0/107 [00:00<?, ? examples/s]

In [ ]:
with open("../models/train.txt", 'r') as file:
    lines = file.readlines()
    
lines = list(map(lambda x: x.split(), lines))
# Преобразование данных в формат, подходящий для загрузки в datasets
dataset_dict = {'text': lines}
dataset_dict

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, AutoTokenizer

model_name = "tiiuae/falcon-7b"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    trust_remote_code=True
)
model.config.use_cache = False



Loading checkpoint shards: 100%|██████████| 2/2 [00:18<00:00,  9.15s/it]


In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

In [4]:
from peft import LoraConfig

lora_alpha = 16
lora_dropout = 0.1
lora_r = 64

peft_config = LoraConfig(
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    r=lora_r,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "query_key_value",
        "dense",
        "dense_h_to_4h",
        "dense_4h_to_h",
    ]
)

In [5]:
from transformers import TrainingArguments

output_dir = "./falcon"
per_device_train_batch_size = 4
gradient_accumulation_steps = 4
optim = "paged_adamw_32bit"
save_steps = 10
logging_steps = 10
learning_rate = 2e-4
max_grad_norm = 0.3
max_steps = 150
warmup_ratio = 0.03
lr_scheduler_type = "constant"

training_arguments = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    optim=optim,
    save_steps=save_steps,
    logging_steps=logging_steps,
    learning_rate=learning_rate,
    fp16=True,
    max_grad_norm=max_grad_norm,
    max_steps=max_steps,
    warmup_ratio=warmup_ratio,
    group_by_length=True,
    lr_scheduler_type=lr_scheduler_type,
    gradient_checkpointing=True,
)

In [6]:
from trl import SFTTrainer

max_seq_length = 512

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    tokenizer=tokenizer,
    args=training_arguments,
)

Map: 100%|██████████| 107/107 [00:00<00:00, 6040.90 examples/s]


In [7]:
for name, module in trainer.model.named_modules():
    if "norm" in name:
        module = module.to(torch.float32)

In [8]:
torch.cuda.is_available()

True

In [9]:
trainer.train()

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
wandb: Currently logged in as: blackbumerrus. Use `wandb login --relogin` to force relogin
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallel

You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss
10,1.340000
20,0.531800
30,0.407900
40,0.361000
50,0.324100
60,0.287700
70,0.241700
80,0.204200
90,0.169000
100,0.142700


TrainOutput(global_step=150, training_loss=0.3028779145081838, metrics={'train_runtime': 618.471, 'train_samples_per_second': 3.881, 'train_steps_per_second': 0.243, 'total_flos': 6984814311202560.0, 'train_loss': 0.3028779145081838, 'epoch': 22.22})

In [11]:
import transformers

In [18]:
pipeline = transformers.pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    device_map="auto",
)
sequences = pipeline(
   "Tell joke about a hedgehog",
    max_length=200,
    do_sample=True,
    top_k=10,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
)
for seq in sequences:
    print(f"Result: {seq['generated_text']}")

Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.


Result: Tell joke about a hedgehog, a fox, and a chicken. What kind of animal do you get when you cross a hedgehog with a.
Joke
Tell a joke about a hedgehog, a fox, and a chicken. What kind of animal do you get when you cross a hedgehog with a fox?
- A foxhog! /
- What kind of animal do you get when you cross an elephant, a pig, and a hippopotamus? /
- An elephant! /
- What kind of animal do you get when you cross a fox, a pig, and a frog? /
- A porky frog! /
- What kind of animal do you get when you cross a fox, a pig, and a rabbit? /
- A porky rabbit! /
- Why can’t you take a pig to the dentist? /
- He bristles at the idea. /

